In [36]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
import pandas as pd
import numpy as np

In [38]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [39]:
path = '/workspaces/crmprtd/update_sheet/'
df = pd.read_excel(path + '20250918-Metadata-AllRequiredChanges.xlsx', header = 1)   # pandas automatically uses openpyxl
df_name = df[
    (df["ISSUE"].str.strip() == "Name") &
    (df["Network ID"] != 11)
]
df_name_new =  df_name[['Station ID', 'Unique Names', 'Native ID']].reset_index(drop=True)

df_name


,ISSUE,Network ID,Network Name,Native ID,Station ID,Unique Names,Unique Location (String),Unique Records,Uniq Obs Freqs,Variables,...,SITE TYPE,OWNER,FIRE CENTRE,FIRE ZONE,LATITUDE,LONGITUDE,ELEVATION,Unnamed: 50,Unnamed: 51,Unnamed: 52
313,Name,12.0,BC-FWx,127,1694.0,TUMBLER(DENISON) -> TUMBLER HUB,"120.9341 W, 55.0274 N, Elev. 942 m",1988-12-14 to 2025-07-22,Hourly,Dew Point Temperature|Precipitation Amount|Pre...,...,Active,Auto-Caller,AXIOM/F6,Hub Station,BC Wildfire Service,Prince George Fire Centre,Dawson Creek Zone,55.0274,-120.934,932
329,Name,12.0,BC-FWx,164,1727.0,EUTSUK,"126.17 W, 53.27 N, Elev. 1036 m",1998-04-30 to 2007-09-13,Hourly,Precipitation Amount|Precipitation Climatology...,...,Archived,Hourly Polling,MCC550C,Weather Station - MB,BC Wildfire Service,Northwest Fire Centre,Nadina Zone (Lakes),53.27,-126.17,1036
350,Name,12.0,BC-FWx,212,1769.0,PLACE LK. -> PLACE LAKE,"122 W, 51.8167 N, Elev. 1065 m",1988-11-01 to 2025-07-22,Hourly,Dew Point Temperature|Precipitation Amount|Pre...,...,Active,Auto-Caller,AXIOM/F6,Weather Station - UHF,BC Wildfire Service,Cariboo Fire Centre,Central Cariboo Zone (Williams Lake),51.8167,-122,1065
443,Name,12.0,BC-FWx,346,1891.0,SALMON ARM,"119.235 W, 50.685 N, Elev. 527 m",1989-10-05 to 2023-04-18,Hourly,Dew Point Temperature|Precipitation Amount|Pre...,...,Archived,Auto-Caller,FWS12S,Weather Station - UHF,BC Wildfire Service,Kamloops Fire Centre,Vernon Zone (Vernon),50.685,-119.235,527
458,Name,12.0,BC-FWx,363,1908.0,REVELSTOKE FS -> REVELSTOKE,"118.2172 W, 51.0603 N, Elev. 680 m",1995-06-29 to 2025-07-22,Hourly,Dew Point Temperature|Precipitation Amount|Pre...,...,Active,Auto-Caller,AXIOM/F6,Weather Station - Cell,BC Wildfire Service,Southeast Fire Centre,Columbia Zone,51.0603,-118.2172,680
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
929,Name,5.0,BCH-GSO-HMP,SNC,12349.0,Site C North Camp,"120.902 W, 56.203 N, Elev. 583 m",2017-01-01 to 2020-07-01,Unspecified,Air Pressure (Point)|Downwelling Shortwave Rad...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
931,Name,5.0,BCH-GSO-HMP,STA,2505.0,Stave R. ab Stave Lk -> Stave River above Stav...,"122.3219444 W, 49.55611111 N, Elev. 330 m",1960-01-02 to 2024-09-20,Daily,Precipitation Amount|Precipitation Climatology...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
935,Name,5.0,BCH-GSO-HMP,SGL,2502.0,Sugar Lake Res. @ Outlet -> Sigar Lake Resevoi...,"118.53 W, 50.35 N, Elev. 675 m",1999-01-03 to 2024-09-13,Daily,Precipitation Amount|Precipitation Climatology...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
940,Name,5.0,BCH-GSO-HMP,WAH,2507.0,Wahleach (Jones) Res. -> Wahleach Resevoir (Jo...,"121.6186111 W, 49.23194444 N, Elev. 641 m",1960-01-02 to 2025-07-15,Daily,Precipitation Amount|Precipitation Climatology...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
def split_station_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    
df_name = df_name_new.copy()

df_name[['old_name', 'new_name', 'n_names']] = (
    df_name['Unique Names'].apply(split_station_name)
)

df_name = df_name.drop(columns= 'Unique Names')

In [41]:
df_name

,Station ID,Native ID,old_name,new_name,n_names
0,1694.0,127,TUMBLER(DENISON),TUMBLER HUB,2
1,1727.0,164,EUTSUK,EUTSUK,1
2,1769.0,212,PLACE LK.,PLACE LAKE,2
3,1891.0,346,SALMON ARM,SALMON ARM,1
4,1908.0,363,REVELSTOKE FS,REVELSTOKE,2
...,...,...,...,...,...
63,12349.0,SNC,Site C North Camp,Site C North Camp,1
64,2505.0,STA,Stave R. ab Stave Lk,Stave River above Stave Lake,2
65,2502.0,SGL,Sugar Lake Res. @ Outlet,Sigar Lake Resevoir at the Outlet,2
66,2507.0,WAH,Wahleach (Jones) Res.,Wahleach Resevoir (Jones Lake),2


In [42]:
check_sql = sa.text("""
SELECT
    station_id,
    station_name
FROM meta_history
WHERE station_id = :station_id
""")

In [43]:
import numpy as np

with engine.begin() as conn:
    for _, row in df_name.iterrows():

        station_id = int(row["Station ID"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---

        if db_row.station_name != old_name:
            print(
                f"⚠️ Station {station_id} (DB, XLS values differ)\n"
                f"   DB : station name is {db_row.station_name}\n"
                f"   XLS: station name is {old_name}"
            )
        else:
            print(
                f"✅ Station {station_id}, {db_row.station_name} the names match "
            )


⚠️ Station 1694 (DB, XLS values differ)
   DB : station name is TUMBLER HUB
   XLS: station name is TUMBLER(DENISON)
✅ Station 1727, EUTSUK the names match 
⚠️ Station 1769 (DB, XLS values differ)
   DB : station name is PLACE LAKE
   XLS: station name is PLACE LK.
✅ Station 1891, SALMON ARM the names match 
⚠️ Station 1908 (DB, XLS values differ)
   DB : station name is REVELSTOKE
   XLS: station name is REVELSTOKE FS
⚠️ Station 1926 (DB, XLS values differ)
   DB : station name is FALLS CREEK
   XLS: station name is FALLS CK
✅ Station 1927, HOWSER the names match 
⚠️ Station 1934 (DB, XLS values differ)
   DB : station name is EIGHT MILE
   XLS: station name is 8 MILE
✅ Station 1940, CRESTON the names match 
⚠️ Station 1957 (DB, XLS values differ)
   DB : station name is TOBY HUB
   XLS: station name is TOBY
⚠️ Station 2236 (DB, XLS values differ)
   DB : station name is Bigattini
   XLS: station name is NEGRO CK
✅ Station 2302, TURNER _Tweedsmuir the names match 
⚠️ Station 2304 (DB,

In [44]:
update_sql = sa.text("""
UPDATE meta_history
SET
    station_name  = :new_name
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_name.iterrows():

        station_id = int(row["Station ID"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_name": new_name,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_name}) → "
                f"({new_name})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")

✅ Updated station 1694: (TUMBLER(DENISON)) → (TUMBLER HUB)
✅ Updated station 1727: (EUTSUK) → (EUTSUK)
✅ Updated station 1769: (PLACE LK.) → (PLACE LAKE)
✅ Updated station 1891: (SALMON ARM) → (SALMON ARM)
✅ Updated station 1908: (REVELSTOKE FS) → (REVELSTOKE)
✅ Updated station 1926: (FALLS CK) → (FALLS CREEK)
✅ Updated station 1927: (HOWSER) → (HOWSER)
✅ Updated station 1934: (8 MILE) → (EIGHT MILE)
✅ Updated station 1940: (CRESTON) → (CRESTON)
✅ Updated station 1957: (TOBY) → (TOBY HUB)
✅ Updated station 2236: (NEGRO CK) → (Bigattini)
✅ Updated station 2302: (TURNER _Tweedsmuir) → (TURNER _Tweedsmuir)
✅ Updated station 2304: (GOLDSTREAM II) → (GOLDSTREAM 2)
✅ Updated station 2311: (ANDERSON) → (ANDERSON)
✅ Updated station 2328: (OELRICH) → (OELRICH)
✅ Updated station 2332: (SASKUM) → (SASKUM)
✅ Updated station 11001: (NORTH SLOQUET) → (NORTH SLOQUET)
✅ Updated station 12018: (NARROWS INLET) → (NARROWS INLET)
✅ Updated station 12337: (INKANEEP EXT) → (INKANEEP EXT)
✅ Updated station 1

In [45]:
import numpy as np

with engine.begin() as conn:
    for _, row in df_name.iterrows():

        station_id = int(row["Station ID"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---

        if db_row.station_name != new_name:
            print(
                f"⚠️ Station {station_id} (DB, XLS values differ)\n"
                f"   DB : station name is {db_row.station_name}\n"
                f"   XLS: station name is {new_name}"
            )
        else:
            print(
                f"✅ Station {station_id}, {db_row.station_name} the names match "
            )


✅ Station 1694, TUMBLER HUB the names match 
✅ Station 1727, EUTSUK the names match 
✅ Station 1769, PLACE LAKE the names match 
✅ Station 1891, SALMON ARM the names match 
✅ Station 1908, REVELSTOKE the names match 
✅ Station 1926, FALLS CREEK the names match 
✅ Station 1927, HOWSER the names match 
✅ Station 1934, EIGHT MILE the names match 
✅ Station 1940, CRESTON the names match 
✅ Station 1957, TOBY HUB the names match 
✅ Station 2236, Bigattini the names match 
✅ Station 2302, TURNER _Tweedsmuir the names match 
✅ Station 2304, GOLDSTREAM 2 the names match 
✅ Station 2311, ANDERSON the names match 
✅ Station 2328, OELRICH the names match 
✅ Station 2332, SASKUM the names match 
✅ Station 11001, NORTH SLOQUET the names match 
✅ Station 12018, NARROWS INLET the names match 
✅ Station 12337, INKANEEP EXT the names match 
✅ Station 12524, APEX EXT the names match 
✅ Station 12622, MERRITT 2 HUB the names match 
✅ Station 12350, 85th Avenue the names match 
✅ Station 12343, Attachie F